In [1]:
# read file
with open(r"C:\Users\joly-\Github\HUMAN\elections\dublin_core_data\export_dnpp_DC.xml") as f:
    content = f.read()
print(content)

relation: https://dnpprepo.ub.rug.nl/88759/
title: Aanpassen aan de draagkracht van de aarde
date: 2025
type: Verkiezingsprogramma's
type: NonPeerReviewed
format: text
language: nl
identifier: https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%20Partijprogramma%202025.pdf
identifier:    De Groenen  (2025) Aanpassen aan de draagkracht van de aarde.  [Verkiezingsprogramma's]     

relation: https://dnpprepo.ub.rug.nl/88739/
title: BBB levert
date: 2025
type: Verkiezingsprogramma's
type: NonPeerReviewed
format: text
language: nl
identifier: https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verkiezingsprogramma%20TK2025.pdf
format: text
language: nl
identifier: https://dnpprepo.ub.rug.nl/88739/1/BBB%20Verkiezingsprogramma%20TK2025%20%28Concept%29.pdf
identifier:    BoerBurgerBeweging  (2025) BBB levert.  [Verkiezingsprogramma's]     

relation: https://dnpprepo.ub.rug.nl/88765/
title: Bouwen op vertrouwen
date: 2025
type: Verkiezingsprogramma's
type: NonPeerReviewed
format: text
language: nl
identifier: 

In [11]:
import json
import re

def parse_metadata_file(path):
    records = []
    current = {}

    url_pattern = re.compile(r"^https?://")

    def commit_record():
        nonlocal current
        if current:
            # Move last non-URL identifier to separate field
            if "identifier" in current:
                urls = []
                citation = None

                for item in current["identifier"]:
                    if url_pattern.match(item):
                        urls.append(item)
                    else:
                        citation = item  # last non-URL wins

                if urls:
                    current["identifier_urls"] = urls
                if citation:
                    current["identifier_citation"] = citation

                del current["identifier"]

            records.append(current)
            current = {}

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("relation:"):
                commit_record()
                current["relation"] = line.split(":", 1)[1].strip()
                continue

            if ":" in line:
                key, value = line.split(":", 1)
                key = key.strip()
                value = value.strip()

                # Collect repeated keys
                if key in current:
                    if not isinstance(current[key], list):
                        current[key] = [current[key]]
                    current[key].append(value)
                else:
                    current[key] = value

    commit_record()
    return records


# Example usage:
input_file = r"C:\Users\joly-\Github\HUMAN\elections\dublin_core_data\export_dnpp_DC.xml"
output_file = r"C:\Users\joly-\Github\HUMAN\elections\dublin_core_data\export_dnpp_DC.json"

data = parse_metadata_file(input_file)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Done! JSON written to", output_file)

Done! JSON written to C:\Users\joly-\Github\HUMAN\elections\dublin_core_data\export_dnpp_DC.json
